In [1]:
"""
M7U4 — Data Quality Queries
============================
Eight Cypher queries answering the data quality questions defined in the
assignment. Each query is documented with:
  - The quality dimension it tests
  - The nodes/relationships involved
  - What the query returns
  - How the results should be interpreted
"""

import os
from pathlib import Path
import pandas as pd
from neo4j import GraphDatabase
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / "results"
QUERIES_DIR = PROJECT_ROOT / "queries"
RESULTS_DIR.mkdir(exist_ok=True)
QUERIES_DIR.mkdir(exist_ok=True)

load_dotenv(PROJECT_ROOT / ".env")
driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))
)

def run_query(cypher, **params):
    """Run a Cypher query and return results as a pandas DataFrame."""
    with driver.session() as session:
        result = session.run(cypher, **params)
        df = pd.DataFrame([r.data() for r in result])
    return df

def save_query_and_results(q_id, q_name, cypher, df):
    """Save Cypher to queries/Q*.cypher and results to results/Q*.csv."""
    cypher_path = QUERIES_DIR / f"{q_id}_{q_name}.cypher"
    csv_path = RESULTS_DIR / f"{q_id}_{q_name}.csv"
    cypher_path.write_text(cypher.strip() + "\n", encoding="utf-8")
    df.to_csv(csv_path, index=False, encoding="utf-8")
    print(f"  Saved query → {cypher_path.name}")
    print(f"  Saved CSV   → {csv_path.name} ({len(df)} rows)")

print("Setup complete. Ready to run queries.")

Setup complete. Ready to run queries.


In [2]:
Q1_NAME = "elements_without_psets"
Q1_CYPHER = """
// Q1 — Completeness
// Find every :Element node that has no outgoing HAS_PSET relationship.
// Such elements carry no semantic metadata beyond their IFC class and name,
// breaking downstream uses (cost, scheduling, FM handover).

MATCH (e:Element)
WHERE NOT (e)-[:HAS_PSET]->()
RETURN e.GlobalId AS GlobalId,
       e.IfcClass AS IfcClass,
       e.Name     AS Name,
       e.Tag      AS Tag
ORDER BY e.IfcClass, e.Name
"""

df_q1 = run_query(Q1_CYPHER)
print(f"Q1: {len(df_q1)} elements have no property sets.")
save_query_and_results("Q1", Q1_NAME, Q1_CYPHER, df_q1)
df_q1.head(20)

Q1: 41 elements have no property sets.
  Saved query → Q1_elements_without_psets.cypher
  Saved CSV   → Q1_elements_without_psets.csv (41 rows)


,GlobalId,IfcClass,Name,Tag
0,1gtrSK5QnDuxDwygd0EDGO,IfcMember,Stair:Residential - 200mm Max Riser 250mm Trea...,151086
1,34qUFGjJzFKwVWpXe2dTPt,IfcMember,Stair:Residential - 200mm Max Riser 250mm Trea...,151086
2,01KzA4SPn5IOODwLEb5RNY,IfcMember,Stair:Residential - 200mm Max Riser 250mm Trea...,198878
3,37Fy90kSD2PvviizyM7EKl,IfcMember,Stair:Residential - 200mm Max Riser 250mm Trea...,198878
4,1xS3BCk291UvhgP2dvNo1Q,IfcOpeningElement,M_Casement:819mm x 759mm:819mm x 759mm:148607:1,NaN
5,1xS3BCk291UvhgP2dvNopD,IfcOpeningElement,M_Casement:819mm x 759mm:819mm x 759mm:149736:1,NaN
6,1xS3BCk291UvhgP2dvNwSd,IfcOpeningElement,M_Casement:819mm x 759mm:819mm x 759mm:180994:1,NaN
7,1xS3BCk291UvhgP2dvNw49,IfcOpeningElement,M_Casement:819mm x 759mm:819mm x 759mm:181548:1,NaN
8,1xS3BCk291UvhgP2dvNoJ3,IfcOpeningElement,M_Fixed:2800mm x 2410mm:2800mm x 2410mm:147686:1,NaN
9,1xS3BCk291UvhgP2dvNoCx,IfcOpeningElement,M_Fixed:2800mm x 2410mm:2800mm x 2410mm:149278:1,NaN


In [3]:
Q2_NAME = "doors_missing_firerating"
Q2_CYPHER = """
// Q2 — Completeness (compliance-critical)
// Doors should carry Pset_DoorCommon.FireRating for compartmentation checks.
// We split the result into two states:
//   "Missing" — no FireRating property exists at all
//   "Empty"   — the property exists but its value is blank
// Both states fail a fire-safety compliance check.

MATCH (d:Element:IfcDoor)
OPTIONAL MATCH (d)-[:HAS_PSET]->(:PropertySet)-[:HAS_PROPERTY]->(p:Property {Name: 'FireRating'})
WITH d, p
WHERE p IS NULL OR p.IsEmpty = true
RETURN d.GlobalId AS GlobalId,
       d.Name     AS DoorName,
       d.Tag      AS Tag,
       CASE WHEN p IS NULL THEN 'Missing' ELSE 'Empty' END AS FireRatingStatus,
       p.Value    AS RawValue
ORDER BY FireRatingStatus, d.Name
"""

df_q2 = run_query(Q2_CYPHER)
print(f"Q2: {len(df_q2)} doors have missing or empty FireRating.")
save_query_and_results("Q2", Q2_NAME, Q2_CYPHER, df_q2)
df_q2

Q2: 0 doors have missing or empty FireRating.
  Saved query → Q2_doors_missing_firerating.cypher
  Saved CSV   → Q2_doors_missing_firerating.csv (0 rows)


""


In [4]:
Q3_NAME = "elements_without_storey"
Q3_CYPHER = """
// Q3 — Consistency (spatial integrity)
// Every physical element should be spatially contained by a Storey
// (via IfcRelContainedInSpatialStructure → CONTAINS in our graph).
// Orphaned elements break automated quantity take-offs, energy simulations,
// and any storey-based dashboard.

MATCH (e:Element)
WHERE NOT EXISTS {
  MATCH (:Storey)-[:CONTAINS]->(e)
}
RETURN e.GlobalId AS GlobalId,
       e.IfcClass AS IfcClass,
       e.Name     AS Name,
       e.Tag      AS Tag
ORDER BY e.IfcClass
"""

df_q3 = run_query(Q3_CYPHER)
print(f"Q3: {len(df_q3)} elements are not assigned to any storey.")
save_query_and_results("Q3", Q3_NAME, Q3_CYPHER, df_q3)
df_q3.head(20)

Q3: 122 elements are not assigned to any storey.
  Saved query → Q3_elements_without_storey.cypher
  Saved CSV   → Q3_elements_without_storey.csv (122 rows)


,GlobalId,IfcClass,Name,Tag
0,0wkEuT1wr1kOyafLY4vyu$,IfcFurnishingElement,M_Tall Cabinet-Single Door(2):800 mm:800 mm:15...,157200
1,0wkEuT1wr1kOyafLY4vy_8,IfcFurnishingElement,M_Tall Cabinet-Single Door(2):800 mm:800 mm:15...,157607
2,0wkEuT1wr1kOyafLY4vy3H,IfcFurnishingElement,M_Tall Cabinet-Single Door(2):800 mm:800 mm:15...,157950
3,0wkEuT1wr1kOyafLY4vy3G,IfcFurnishingElement,M_Tall Cabinet-Single Door(2):800 mm:800 mm:15...,157951
4,0wkEuT1wr1kOyafLY4vy4m,IfcFurnishingElement,M_Tall Cabinet-Single Door(2):800 mm:800 mm:15...,157983
5,0wkEuT1wr1kOyafLY4vy4F,IfcFurnishingElement,M_Tall Cabinet-Single Door(2):800 mm:800 mm:15...,157984
6,0wkEuT1wr1kOyafLY4vy6l,IfcFurnishingElement,M_Tall Cabinet-Single Door(2):800 mm:800 mm:15...,158080
7,0wkEuT1wr1kOyafLY4vy6k,IfcFurnishingElement,M_Tall Cabinet-Single Door(2):800 mm:800 mm:15...,158081
8,0wkEuT1wr1kOyafLY4vyMO,IfcFurnishingElement,M_Base Cabinet-Double Door & 2 Drawer:1000mm:1...,159159
9,0wkEuT1wr1kOyafLY4vyOn,IfcFurnishingElement,M_Base Cabinet-Double Door & 2 Drawer:1000mm:1...,159262


In [5]:
Q4_NAME = "spaces_missing_identity"
Q4_CYPHER = """
// Q4 — Completeness (identity)
// In IFC, IfcSpace.Name typically holds the room number (e.g. "101")
// and IfcSpace.LongName holds the descriptive name (e.g. "Living Room").
// A space missing either cannot be referenced in FM systems or area schedules.

MATCH (s:Space)
WHERE s.Name IS NULL     OR trim(s.Name) = ''
   OR s.LongName IS NULL OR trim(s.LongName) = ''
RETURN s.GlobalId                              AS GlobalId,
       coalesce(s.Name, '<MISSING>')           AS RoomNumber,
       coalesce(s.LongName, '<MISSING>')       AS RoomName,
       s.Description                           AS Description
ORDER BY s.GlobalId
"""

df_q4 = run_query(Q4_CYPHER)
print(f"Q4: {len(df_q4)} spaces are missing their name or number.")
save_query_and_results("Q4", Q4_NAME, Q4_CYPHER, df_q4)
df_q4

Q4: 0 spaces are missing their name or number.
  Saved query → Q4_spaces_missing_identity.cypher
  Saved CSV   → Q4_spaces_missing_identity.csv (0 rows)


""


In [6]:
Q5_NAME = "properties_with_empty_values"
Q5_CYPHER = """
// Q5 — Completeness (the "illusion of structured data")
// Lecture 2: "Exporting a Pset does not mean exporting useful information.
// Empty placeholders create the illusion of structured data."
// We pre-computed IsEmpty=true at load time for any property whose value is
// null or whitespace-only, so this query is now a direct lookup.

MATCH (e:Element)-[:HAS_PSET]->(ps:PropertySet)-[:HAS_PROPERTY]->(p:Property)
WHERE p.IsEmpty = true
RETURN e.GlobalId AS ElementGlobalId,
       e.IfcClass AS IfcClass,
       e.Name     AS ElementName,
       ps.Name    AS PropertySet,
       p.Name     AS PropertyName,
       p.DataType AS DataType
ORDER BY e.IfcClass, ps.Name, p.Name
"""

df_q5 = run_query(Q5_CYPHER)
print(f"Q5: {len(df_q5)} property instances have empty values.")
save_query_and_results("Q5", Q5_NAME, Q5_CYPHER, df_q5)
df_q5.head(20)

Q5: 452 property instances have empty values.
  Saved query → Q5_properties_with_empty_values.cypher
  Saved CSV   → Q5_properties_with_empty_values.csv (452 rows)


,ElementGlobalId,IfcClass,ElementName,PropertySet,PropertyName,DataType
0,2OrWItJ6zAwBNp0OUxK_l8,IfcBeam,M_W-Wide Flange:W310X60:W310X60:207325,PSet_Revit_Type_Identity Data,Assembly Code,str
1,2OrWItJ6zAwBNp0OUxK$8W,IfcBeam,M_W-Wide Flange:W410X60:W410X60:208949,PSet_Revit_Type_Identity Data,Assembly Code,str
2,2OrWItJ6zAwBNp0OUxK$Bq,IfcBeam,M_W-Wide Flange:W410X60:W410X60:209121,PSet_Revit_Type_Identity Data,Assembly Code,str
3,2OrWItJ6zAwBNp0OUxK$CR,IfcBeam,M_W-Wide Flange:W410X60:W410X60:209166,PSet_Revit_Type_Identity Data,Assembly Code,str
4,2OrWItJ6zAwBNp0OUxK$Dv,IfcBeam,M_W-Wide Flange:W310X60:W310X60:209260,PSet_Revit_Type_Identity Data,Assembly Code,str
5,2OrWItJ6zAwBNp0OUxK$Du,IfcBeam,M_W-Wide Flange:W410X60:W410X60:209261,PSet_Revit_Type_Identity Data,Assembly Code,str
6,2OrWItJ6zAwBNp0OUxK$Dx,IfcBeam,M_W-Wide Flange:W410X60:W410X60:209262,PSet_Revit_Type_Identity Data,Assembly Code,str
7,2OrWItJ6zAwBNp0OUxK$Dw,IfcBeam,M_W-Wide Flange:W410X60:W410X60:209263,PSet_Revit_Type_Identity Data,Assembly Code,str
8,2OrWItJ6zAwBNp0OUxK_l8,IfcBeam,M_W-Wide Flange:W310X60:W310X60:207325,PSet_Revit_Type_Identity Data,Assembly Description,str
9,2OrWItJ6zAwBNp0OUxK$8W,IfcBeam,M_W-Wide Flange:W410X60:W410X60:208949,PSet_Revit_Type_Identity Data,Assembly Description,str


In [7]:
Q6_NAME = "incompleteness_by_category"
Q6_CYPHER = """
// Q6 — Completeness, aggregated by IFC category
// For every IFC class:
//   TotalElements    — how many of them are in the model
//   ElementsNoPset   — how many have zero property sets at all
//   TotalProperties  — sum of all property instances across the class
//   EmptyProperties  — how many of those are empty
//   EmptyRatio       — EmptyProperties / TotalProperties (0 if no props)
// Used to prioritise which categories to remediate first.

MATCH (e:Element)
OPTIONAL MATCH (e)-[:HAS_PSET]->(:PropertySet)-[:HAS_PROPERTY]->(p:Property)
WITH e.IfcClass AS Category,
     e,
     p
WITH Category,
     count(DISTINCT e) AS TotalElements,
     sum(CASE WHEN p IS NULL THEN 1 ELSE 0 END) AS ElementsNoProperties,
     count(p) AS TotalProperties,
     sum(CASE WHEN p.IsEmpty = true THEN 1 ELSE 0 END) AS EmptyProperties
RETURN Category,
       TotalElements,
       TotalProperties,
       EmptyProperties,
       CASE WHEN TotalProperties = 0 THEN 0.0
            ELSE round(toFloat(EmptyProperties) / TotalProperties, 3)
       END AS EmptyRatio
ORDER BY EmptyProperties DESC, EmptyRatio DESC
"""

df_q6 = run_query(Q6_CYPHER)
print(f"Q6: ranked {len(df_q6)} IFC categories by incompleteness.")
save_query_and_results("Q6", Q6_NAME, Q6_CYPHER, df_q6)
df_q6

Q6: ranked 15 IFC categories by incompleteness.
  Saved query → Q6_incompleteness_by_category.cypher
  Saved CSV   → Q6_incompleteness_by_category.csv (15 rows)


,Category,TotalElements,TotalProperties,EmptyProperties,EmptyRatio
0,IfcFurnishingElement,61,3011,122,0.041
1,IfcWallStandardCase,56,3344,112,0.033
2,IfcWindow,24,1480,50,0.034
3,IfcSlab,21,1100,40,0.036
4,IfcOpeningElement,50,742,28,0.038
5,IfcDoor,14,890,28,0.031
6,IfcCovering,13,624,26,0.042
7,IfcBeam,8,696,16,0.023
8,IfcFooting,7,365,14,0.038
9,IfcRailing,4,200,8,0.040


In [8]:
Q7_NAME = "duplicate_identifiers_and_names"

# Q7 is two queries combined into one report — a uniqueness check on GlobalIds
# (which a constraint should already prevent) and a uniqueness analysis of Name.

Q7A_CYPHER = """
// Q7a — Uniqueness (GlobalId)
// IFC GlobalIds are required to be globally unique. A duplicate here would
// indicate a serious modelling or export defect.
MATCH (e:Element)
WITH e.GlobalId AS GlobalId, count(*) AS Occurrences, collect(e.IfcClass) AS Classes
WHERE Occurrences > 1
RETURN GlobalId, Occurrences, Classes
"""

Q7B_CYPHER = """
// Q7b — Uniqueness (Name within IFC class)
// In IFC it is common for many instances to share a Name (e.g. "Basic Wall:200mm")
// because the Name carries the type/family rather than an instance identifier.
// High duplication counts are not strictly errors but signal weak individual
// identification — a problem for asset tagging and FM lookup.
MATCH (e:Element)
WHERE e.Name IS NOT NULL AND trim(e.Name) <> ''
WITH e.IfcClass AS IfcClass, e.Name AS Name,
     count(*) AS Occurrences, collect(e.GlobalId)[..5] AS SampleGlobalIds
WHERE Occurrences > 1
RETURN IfcClass, Name, Occurrences, SampleGlobalIds
ORDER BY Occurrences DESC, IfcClass, Name
"""

df_q7a = run_query(Q7A_CYPHER)
df_q7b = run_query(Q7B_CYPHER)
print(f"Q7a: {len(df_q7a)} duplicate GlobalIds (should be 0).")
print(f"Q7b: {len(df_q7b)} Name+IfcClass combinations shared by multiple instances.")

# Save both
(QUERIES_DIR / "Q7a_duplicate_globalids.cypher").write_text(Q7A_CYPHER.strip()+"\n", encoding="utf-8")
(QUERIES_DIR / "Q7b_duplicate_names.cypher").write_text(Q7B_CYPHER.strip()+"\n", encoding="utf-8")
df_q7a.to_csv(RESULTS_DIR / "Q7a_duplicate_globalids.csv", index=False)
df_q7b.to_csv(RESULTS_DIR / "Q7b_duplicate_names.csv", index=False)
print("  Saved Q7a + Q7b queries and CSVs.")

print("\nQ7a (duplicate GlobalIds):")
display(df_q7a)
print("\nQ7b (duplicate Names within class) — top 20:")
df_q7b.head(20)

Q7a: 0 duplicate GlobalIds (should be 0).
Q7b: 0 Name+IfcClass combinations shared by multiple instances.
  Saved Q7a + Q7b queries and CSVs.

Q7a (duplicate GlobalIds):


""



Q7b (duplicate Names within class) — top 20:


""


In [9]:
Q8_NAME = "values_outside_permitted_set"
Q8_CYPHER = """
// Q8 — Validity (controlled vocabulary)
// Two checks against known controlled vocabularies:
//   FireRating  — numeric minutes (30/60/90/120/180/240) or "FDxx" codes
//   IsExternal  — boolean (must be true/false, case-insensitive)
// Anything outside these sets is a validity failure: the value exists but is
// not interpretable by a downstream compliance engine.

MATCH (e:Element)-[:HAS_PSET]->(:PropertySet)-[:HAS_PROPERTY]->(p:Property)
WHERE p.IsEmpty = false
  AND (
    (p.Name = 'FireRating'
       AND NOT p.Value IN ['30','60','90','120','180','240',
                           'FD30','FD60','FD90','FD120','FD180','FD240'])
    OR
    (p.Name = 'IsExternal'
       AND NOT toLower(p.Value) IN ['true','false'])
  )
RETURN e.GlobalId AS ElementGlobalId,
       e.IfcClass AS IfcClass,
       e.Name     AS ElementName,
       p.Name     AS PropertyName,
       p.Value    AS RawValue,
       p.DataType AS DataType
ORDER BY p.Name, p.Value
"""

df_q8 = run_query(Q8_CYPHER)
print(f"Q8: {len(df_q8)} property values fall outside the permitted set.")
save_query_and_results("Q8", Q8_NAME, Q8_CYPHER, df_q8)
df_q8.head(20)

Q8: 62 property values fall outside the permitted set.
  Saved query → Q8_values_outside_permitted_set.cypher
  Saved CSV   → Q8_values_outside_permitted_set.csv (62 rows)


,ElementGlobalId,IfcClass,ElementName,PropertyName,RawValue,DataType
0,1hOSvn6df7F8_7GcBWlRGQ,IfcDoor,M_Single-Flush:1250mm x 2010mm:1250mm x 2010mm...,FireRating,Fire Rating,str
1,1hOSvn6df7F8_7GcBWlRH8,IfcDoor,M_Single-Flush:1250mm x 2010mm:1250mm x 2010mm...,FireRating,Fire Rating,str
2,1hOSvn6df7F8_7GcBWlS8Z,IfcDoor,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:150173,FireRating,Fire Rating,str
3,1hOSvn6df7F8_7GcBWlS9F,IfcDoor,M_Single-Flush:0762 x 2032mm:0762 x 2032mm:150257,FireRating,Fire Rating,str
4,1hOSvn6df7F8_7GcBWlSFK,IfcDoor,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:150378,FireRating,Fire Rating,str
5,1hOSvn6df7F8_7GcBWlSDm,IfcDoor,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:150478,FireRating,Fire Rating,str
6,2OBrcmyk58NupXoVOHUuXp,IfcDoor,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:159734,FireRating,Fire Rating,str
7,2OBrcmyk58NupXoVOHUvVV,IfcDoor,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:159834,FireRating,Fire Rating,str
8,2OBrcmyk58NupXoVOHUvR4,IfcDoor,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:160065,FireRating,Fire Rating,str
9,2OBrcmyk58NupXoVOHUvPL,IfcDoor,M_Single-Flush:0864 x 2032mm:0864 x 2032mm:160208,FireRating,Fire Rating,str


In [10]:
summary = pd.DataFrame([
    {"Question": "Q1", "Dimension": "Completeness",         "Issue": "Elements with no property sets",       "Findings": len(df_q1)},
    {"Question": "Q2", "Dimension": "Completeness",         "Issue": "Doors missing FireRating",             "Findings": len(df_q2)},
    {"Question": "Q3", "Dimension": "Consistency (spatial)","Issue": "Elements not in any storey",            "Findings": len(df_q3)},
    {"Question": "Q4", "Dimension": "Completeness (ID)",    "Issue": "Spaces missing name/number",            "Findings": len(df_q4)},
    {"Question": "Q5", "Dimension": "Completeness",         "Issue": "Properties with empty values",         "Findings": len(df_q5)},
    {"Question": "Q6", "Dimension": "Completeness (agg.)",  "Issue": "Incompleteness ranked by category",     "Findings": f"{len(df_q6)} categories ranked"},
    {"Question": "Q7", "Dimension": "Uniqueness",           "Issue": "Duplicate GlobalIds | shared Names",    "Findings": f"{len(df_q7a)} | {len(df_q7b)}"},
    {"Question": "Q8", "Dimension": "Validity",             "Issue": "Values outside permitted set",         "Findings": len(df_q8)},
])

summary.to_csv(RESULTS_DIR / "00_summary.csv", index=False)
print("Saved results/00_summary.csv\n")
summary

Saved results/00_summary.csv



,Question,Dimension,Issue,Findings
0,Q1,Completeness,Elements with no property sets,41
1,Q2,Completeness,Doors missing FireRating,0
2,Q3,Consistency (spatial),Elements not in any storey,122
3,Q4,Completeness (ID),Spaces missing name/number,0
4,Q5,Completeness,Properties with empty values,452
5,Q6,Completeness (agg.),Incompleteness ranked by category,15 categories ranked
6,Q7,Uniqueness,Duplicate GlobalIds | shared Names,0 | 0
7,Q8,Validity,Values outside permitted set,62


In [11]:
driver.close()
print("Driver closed. All eight queries complete.")

Driver closed. All eight queries complete.
